<a href="https://colab.research.google.com/github/naubergois/curso-yolo-colab/blob/main/notebooks/03-rastreamento-e-contagem.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 3. Rastreamento e contagem de pessoas

Detectar diz *tem uma pessoa aqui*. Rastrear diz *é a mesma pessoa do quadro anterior*. O rastreador (ByteTrack, que já vem no Ultralytics) liga as caixas de um quadro às do próximo e dá a cada pessoa um **ID** que se mantém enquanto ela estiver em cena.

Com IDs estáveis dá para contar gente de verdade: quantas pessoas diferentes passaram e quantas cruzaram uma linha. O vídeo é o clássico `vtest.avi` do OpenCV, um pátio de universidade filmado do alto.

In [ ]:
!pip install -q ultralytics lap

In [ ]:
import base64
import subprocess
import urllib.request
from pathlib import Path

from IPython.display import HTML, display


def baixar(url, destino):
    if not Path(destino).exists():
        urllib.request.urlretrieve(url, destino)
    print(f"{destino}: {Path(destino).stat().st_size / 1e6:.1f} MB")
    return destino


def mostrar_video(caminho, largura=720):
    """Converte para H.264 (o navegador não toca mp4v) e embute no notebook."""
    web = str(Path(caminho).with_suffix("")) + "_web.mp4"
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", caminho,
         "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart", web],
        check=True,
    )
    dados = base64.b64encode(Path(web).read_bytes()).decode()
    display(HTML(f'<video width="{largura}" controls autoplay muted loop>'
                 f'<source src="data:video/mp4;base64,{dados}" type="video/mp4"></video>'))
    return web


def baixar_para_maquina(caminho):
    try:
        from google.colab import files
        files.download(caminho)
    except ImportError:
        print(f"Fora do Colab: o arquivo está em {Path(caminho).resolve()}")

In [ ]:
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
video = baixar("https://raw.githubusercontent.com/opencv/opencv/master/samples/data/vtest.avi", "calcada.avi")

## Rastrear e contar quem cruza a linha

O segredo está em `model.track(..., persist=True)`: o `persist` faz o rastreador lembrar dos IDs entre uma chamada e outra. `classes=[0]` limita a pessoas.

Para contar cruzamentos, guardamos a posição horizontal do centro de cada ID. Quando ela troca de lado da linha vertical, contamos uma vez aquele ID.

In [ ]:
MAX_QUADROS = 200

cap = cv2.VideoCapture(video)
fps = int(cap.get(cv2.CAP_PROP_FPS)) or 10
largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
altura = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
LINHA_X = largura // 2

out = cv2.VideoWriter("calcada_rastreada.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (largura, altura))

ids_vistos = set()
ultimo_x = {}
cruzaram = set()
por_quadro = []
exemplos = []

for i in range(MAX_QUADROS):
    ok, frame = cap.read()
    if not ok:
        break
    r = model.track(frame, persist=True, classes=[0], verbose=False)[0]
    anotado = r.plot()

    if r.boxes.id is not None:
        ids = r.boxes.id.int().tolist()
        centros_x = r.boxes.xywh[:, 0].tolist()
        for pid, cx in zip(ids, centros_x):
            antes = ultimo_x.get(pid)
            if antes is not None and (antes - LINHA_X) * (cx - LINHA_X) < 0:
                cruzaram.add(pid)
            ultimo_x[pid] = cx
        ids_vistos.update(ids)
        por_quadro.append(len(ids))
    else:
        por_quadro.append(0)

    cv2.line(anotado, (LINHA_X, 0), (LINHA_X, altura), (0, 255, 255), 2)
    cv2.putText(anotado, f"cruzaram: {len(cruzaram)}", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 255, 255), 3)
    out.write(anotado)

    if i in (20, 80, 140, 190):
        exemplos.append(anotado[:, :, ::-1])

cap.release()
out.release()

print(f"Quadros: {len(por_quadro)}")
print(f"Pessoas diferentes (IDs): {len(ids_vistos)}")
print(f"Cruzaram a linha: {len(cruzaram)}")
assert len(ids_vistos) > 0

## Ver quadros de exemplo

In [ ]:
plt.figure(figsize=(14, 10))
for k, img in enumerate(exemplos):
    plt.subplot(2, 2, k + 1)
    plt.imshow(img)
    plt.axis("off")
plt.suptitle(f"{len(ids_vistos)} IDs, {len(cruzaram)} cruzamentos")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 3))
plt.plot(por_quadro)
plt.title("Pessoas rastreadas por quadro")
plt.xlabel("quadro")
plt.grid(True)
plt.show()

## Assistir e baixar

In [ ]:
web = mostrar_video("calcada_rastreada.mp4")
baixar_para_maquina(web)

## O que fica

O ID é o que transforma detecção em contagem. Mas ele não é perfeito: quando uma pessoa some atrás de outra e volta, o rastreador às vezes dá um ID novo. Por isso o número de IDs costuma passar um pouco do número real de pessoas. A contagem por linha sofre menos com isso, já que só conta quem atravessa.